# 10b. Quantization in Practice: AWQ, GPTQ, FP8, GGUF

**Tier:** Training
**Estimated time:** 45 minutes
**Prerequisites:** 08 (fine-tuning & LoRA), 10 (quantization fundamentals)
**Priority:** 🟡 Important — matters once you're loading a specific checkpoint for a specific serving stack. *If skipped, revisit when:* a model card offers you a choice of `-AWQ`, `-GPTQ`, `-FP8`, or `.gguf` files and you need to know which one to click.
**Source material:** Lin et al., *"AWQ: Activation-aware Weight Quantization for LLM Compression and Acceleration"* (MLSys 2024); Frantar et al., *"GPTQ: Accurate Post-Training Quantization for Generative Pre-trained Transformers"* (ICLR 2023); vLLM FP8 W8A8 docs (https://docs.vllm.ai/en/stable/features/quantization/llm_compressor/fp8/); llama.cpp GGUF quantization types.

## What You'll Learn
- Why **AWQ** protects a small set of "salient" weight channels instead of quantizing every weight equally
- What **GPTQ** actually does differently from naive rounding — compensating later weights for the error introduced by earlier ones, using the calibration data's correlation structure
- The **FP8 vs INT4** tradeoff: dynamic range vs. precision, and why FP8 needs no calibration data
- What the **GGUF** tags Ollama shows you (`q4_K_M`, `q8_0`, ...) actually mean, and which quantization format belongs on which serving stack

## Why This Matters
Notebook 10 taught you *why* quantization works — round weights to a coarser grid, trade precision for memory. What it didn't cover is that "quantize to INT4" is not one algorithm. Pull up any popular model on Hugging Face and you'll see a dozen checkpoint variants: `-AWQ`, `-GPTQ`, `-FP8-dynamic`, and a `.gguf` file with `Q4_K_M` in the name. Picking the wrong one either breaks compatibility with your serving stack (vLLM won't load a `.gguf` file the way llama.cpp does) or silently costs you accuracy you didn't need to lose. This notebook is the practical map from "I have a GPU/format/stack" to "here's the checkpoint to pull."


## Recap: the naive baseline, and why it isn't enough

Notebook 10 built round-to-nearest (RTN) quantization: pick a scale so the weight tensor's range maps onto an integer grid, then round every value to its nearest grid point. That's genuinely how far you can get with **no calibration data** — and it's still what a "dynamic" FP8 checkpoint does today.

The problem RTN has is that it treats every weight identically. Real transformer weight matrices don't have a uniform distribution of *importance* — a small number of input channels see activations 10-100x larger than the rest (an empirical finding across essentially every LLM), and the weights multiplying those channels matter far more to the output than an equal-magnitude weight elsewhere. Naive RTN spends its precision budget uniformly and gets the worst of both worlds: the "usual" weights are quantized more finely than they need to be, and the few weights that matter most get the same coarse treatment as everything else.

AWQ and GPTQ are both answers to "spend the precision budget where it matters" — they just spend it differently.

### First: grouping alone already helps

Before either algorithm, one simple change matters a lot: quantizing in small **groups** instead of one scale per row. A single outlier forces one shared scale to stretch across the whole row; splitting the row into groups means only the outlier's own group pays for it.


In [1]:
import numpy as np
import torch

torch.manual_seed(0)
rng = np.random.default_rng(0)

def quantize_int4_per_group(W: torch.Tensor, group_size: int = 128) -> torch.Tensor:
    """Round-to-nearest INT4 quantization, one scale per group of `group_size` input columns.

    This is exactly what vLLM's AWQ/GPTQ kernels and GGUF's K-quants do -- a single
    outlier no longer drags the scale for the entire row, just for its own group.
    """
    out_features, in_features = W.shape
    n_groups = (in_features + group_size - 1) // group_size
    W_deq = torch.empty_like(W)
    qmax = 7  # signed 4-bit: [-8, 7], symmetric so we use +/-7
    for g in range(n_groups):
        lo, hi = g * group_size, min((g + 1) * group_size, in_features)
        chunk = W[:, lo:hi]
        scale = chunk.abs().amax(dim=1, keepdim=True).clamp(min=1e-8) / qmax
        q = torch.clamp(torch.round(chunk / scale), -qmax - 1, qmax)
        W_deq[:, lo:hi] = q * scale
    return W_deq

# A synthetic weight row that looks like a real LLM projection: mostly small values,
# a handful of large ones (the outlier channels AWQ cares about).
IN_FEATURES = 512
W_row = torch.randn(1, IN_FEATURES) * 0.5
outlier_idx = rng.choice(IN_FEATURES, size=8, replace=False)
W_row[0, outlier_idx] *= 12  # a few channels with much larger weights

for group_size in [512, 128, 32]:
    W_deq = quantize_int4_per_group(W_row, group_size=group_size)
    rel_err = (W_row - W_deq).norm() / W_row.norm()
    print(f"group_size={group_size:4d}  relative quantization error: {rel_err:.1%}")


group_size= 512  relative quantization error: 27.1%
group_size= 128  relative quantization error: 23.2%
group_size=  32  relative quantization error: 14.7%


Smaller groups shrink the error because a single group's scale is no longer stretched to cover the whole row's outliers — this is the exact mechanism behind GGUF's `K` quants and the AWQ/GPTQ kernels vLLM ships (both default to `group_size=128`). But group size only helps with outliers *within a group*; it does nothing about *which* weights should get extra protection based on how they're actually used at inference time — which is the problem AWQ targets.

## AWQ: protect the channels the activations actually use

AWQ's insight (Lin et al., 2024) is that weight magnitude alone doesn't tell you which weights matter — **activation magnitude** does. A channel that consistently sees huge activation values will dominate the output for that channel regardless of how "small" its weight looks in isolation, so quantizing it poorly hurts a lot; a channel with tiny activations barely matters even if its weight is unremarkable.

The trick: for a chosen per-channel scale $s$, you can rewrite $Y = XW^T$ as $Y = (X/s) \cdot (sW)^T$ **exactly**, with no approximation yet. Scale *up* the input-channel columns of $W$ that see large activations before quantizing, and scale *down* the matching columns of $X$ by the same factor to keep the math identical. Only $sW$ gets quantized — and here's the part that makes it work: each **output row** picks its own quantization scale from its own max, so scaling up a channel that *isn't already the largest value in most rows* buys those rows a smaller relative rounding error on that channel for free, without inflating their scale. Push $s$ too far, though, and the scaled channel starts dominating every row's max anyway — which is exactly why AWQ searches a small grid of $s$ values instead of picking one blindly.


In [2]:
def quantize_int4_per_row(W: torch.Tensor, qmax: int = 7) -> torch.Tensor:
    """Round-to-nearest INT4, one scale per OUTPUT row (no grouping along the input dim)."""
    scale = W.abs().amax(dim=1, keepdim=True).clamp(min=1e-8) / qmax
    q = torch.clamp(torch.round(W / scale), -qmax - 1, qmax)
    return q * scale

def awq_style_error(W: torch.Tensor, X: torch.Tensor, alpha: float) -> torch.Tensor:
    """Quantize W with AWQ-style protective scaling and measure the OUTPUT error.

    alpha=0 reduces to plain RTN (no protection). alpha>0 scales weight COLUMNS up in
    proportion to how large their activations are, protecting exactly the channels
    AWQ's calibration search identifies as salient -- and compensates by dividing the
    matching activation columns by the same factor (folded into the final division).
    """
    activation_importance = X.abs().mean(dim=0)                       # (in_features,)
    baseline = activation_importance.median().clamp(min=1e-8)          # robust to the outliers
    s = (activation_importance / baseline).clamp(min=1e-8).pow(alpha)  # per-channel scale

    W_scaled = W * s                                                   # protect salient channels
    W_scaled_q = quantize_int4_per_row(W_scaled)
    W_q = W_scaled_q / s                                               # undo the scaling (dequant)

    y_true = X @ W.T
    y_quant = X @ W_q.T
    return (y_true - y_quant).norm() / y_true.norm()

# A layer with MANY output rows: a salient channel only occasionally sets any one row's
# max, so protecting it doesn't automatically blow up every row's quantization scale.
OUT_FEATURES = 128
W_layer = torch.randn(OUT_FEATURES, IN_FEATURES) * 0.5
X_calib = torch.randn(256, IN_FEATURES) * 0.3
X_calib[:, outlier_idx] *= 6  # these 8 channels see 6x larger activations -- the real signal

for alpha in [0.0, 0.1, 0.2, 0.3, 0.5, 0.8, 1.0]:
    err = awq_style_error(W_layer, X_calib, alpha=alpha)
    tag = " (= plain RTN, no protection)" if alpha == 0.0 else ""
    print(f"alpha={alpha:.1f}  output relative error: {err:.2%}{tag}")


alpha=0.0  output relative error: 13.39% (= plain RTN, no protection)
alpha=0.1  output relative error: 12.52%
alpha=0.2  output relative error: 12.32%
alpha=0.3  output relative error: 12.73%
alpha=0.5  output relative error: 15.98%
alpha=0.8  output relative error: 26.51%
alpha=1.0  output relative error: 38.28%


`alpha=0` is plain RTN, evaluated end-to-end through the layer. As `alpha` rises from there, the salient channels get real protection and error drops — until it's pushed too far, the scaled channel starts dominating rows it never used to, and error shoots back up past the unprotected baseline. Real AWQ does a small grid search over `alpha` per layer using a calibration set (a few hundred sequences of general text) and picks whichever minimizes output error; that's the entire "activation-aware" part of the name. No gradient descent, no fine-tuning — just this scale-and-quantize trick applied per layer, which is why AWQ is fast to produce (minutes, not hours) and why it's the default INT4 format vLLM recommends.

## GPTQ: compensate later weights for earlier rounding error

GPTQ takes a different angle: instead of pre-protecting channels, it quantizes weights one column at a time and, after rounding each column, **nudges the not-yet-quantized columns** to compensate for the error just introduced. This is the core idea from Optimal Brain Compression (OBC/OBQ), which GPTQ approximates efficiently: given the layer's activation second-moment matrix ("Hessian") $H = X^TX$ restricted to the still-unquantized columns, the update that best compensates the *remaining* weights for quantizing column $j$ uses the **inverse** of that restricted Hessian, $H^{-1}$:

$$w_i \mathrel{-}= (w_j - \text{round}(w_j)) \cdot \frac{[H^{-1}]_{j,i}}{[H^{-1}]_{j,j}} \quad \text{for every remaining column } i$$

The inverse (not the raw Hessian) is what makes this the *optimal* correction — it accounts for how every remaining column jointly reacts to fixing column $j$, not just their pairwise correlation.


In [3]:
def naive_sequential_quantize(w: torch.Tensor, qmax: int = 7) -> torch.Tensor:
    """Round every weight independently -- no error compensation. The GPTQ baseline."""
    scale = w.abs().max().clamp(min=1e-8) / qmax
    return torch.clamp(torch.round(w / scale), -qmax - 1, qmax) * scale

def gptq_style_quantize(w: torch.Tensor, H: torch.Tensor, qmax: int = 7, damp: float = 1e-2) -> torch.Tensor:
    """Quantize one weight vector column-by-column, compensating the still-unquantized
    columns for each column's rounding error using the INVERSE of the calibration
    Hessian restricted to whatever remains unquantized -- the OBQ/GPTQ update rule.
    Recomputing the inverse at every step (O(n^4) total) is what real GPTQ avoids via
    a Cholesky-based incremental update; here it's spelled out for clarity, not speed.
    """
    w = w.clone()
    n = w.shape[0]
    scale = w.abs().max().clamp(min=1e-8) / qmax
    H = H + damp * H.diag().mean() * torch.eye(n)   # dampen for numerical stability
    remaining = list(range(n))
    for _ in range(n):
        idx = torch.tensor(remaining)
        H_inv = torch.linalg.inv(H[idx][:, idx])     # inverse restricted to what's left
        j = remaining[0]
        q_j = torch.clamp(torch.round(w[j] / scale), -qmax - 1, qmax) * scale
        error = w[j] - q_j
        w[j] = q_j
        denom = H_inv[0, 0]
        for k, idx_k in enumerate(remaining[1:], start=1):
            w[idx_k] -= error * H_inv[0, k] / denom
        remaining.pop(0)
    return w

# Correlated calibration activations -- if columns were independent (H diagonal),
# GPTQ's compensation term would carry no information and it would reduce to naive rounding.
N_FEATS = 64
base = torch.randn(300, N_FEATS // 4)
X_corr = base.repeat_interleave(4, dim=1) + 0.3 * torch.randn(300, N_FEATS)
H = X_corr.T @ X_corr

w_true = torch.randn(N_FEATS) * 0.6

w_naive = naive_sequential_quantize(w_true)
w_gptq = gptq_style_quantize(w_true, H)

y_true = X_corr @ w_true
err_naive = (y_true - X_corr @ w_naive).norm() / y_true.norm()
err_gptq = (y_true - X_corr @ w_gptq).norm() / y_true.norm()
print(f"Naive independent rounding : output relative error {err_naive:.2%}")
print(f"GPTQ-style compensation    : output relative error {err_gptq:.2%}")


Naive independent rounding : output relative error 11.33%
GPTQ-style compensation    : output relative error 6.09%


GPTQ roughly halves the error here because the calibration columns are correlated (`H` is far from diagonal) — the compensation term has real information to work with. With truly independent columns the two methods would tie, which is the real-world reason GPTQ needs a calibration set at all: without correlated activations to compensate against, there's nothing to compensate with.

**AWQ vs GPTQ in one line:** AWQ decides in advance which channels deserve protection and scales them; GPTQ quantizes everything and fixes each mistake using the correlations already present in the rest of the layer. AWQ is faster to produce and slightly more robust across domains; GPTQ has historically eked out marginally lower perplexity at INT4. In practice — per the 2026 tooling landscape — **AWQ has become the default recommendation for new INT4 checkpoints** on vLLM, mainly because its faster Marlin kernel support gives it a throughput edge that erases GPTQ's small quality lead. Both are usually built today via the `llm-compressor` library rather than the original research repos.

## FP8: a different axis entirely

INT4/INT8 spend bits on **precision** — a fixed number of equally-spaced grid points. FP8 (specifically the E4M3 variant used for LLM weights) spends bits on **dynamic range** instead — like INT8, it's 8 bits, but 4 of them are exponent bits, so the representable grid is *denser near zero and sparser at large magnitudes*, mirroring how weight distributions actually look (mostly small, with a long tail).


In [4]:
def quantize_int8_symmetric(w: torch.Tensor) -> torch.Tensor:
    qmax = 127
    scale = w.abs().max().clamp(min=1e-8) / qmax
    return torch.clamp(torch.round(w / scale), -qmax - 1, qmax) * scale

def quantize_fp8_e4m3(w: torch.Tensor) -> torch.Tensor:
    """A minimal E4M3-style float quantizer: 1 sign bit, 4 exponent bits, 3 mantissa bits.
    Real FP8 kernels (vLLM's W8A8) do this in hardware; this is the same rounding rule
    in plain torch so the INT8-vs-FP8 error comparison below is apples-to-apples.
    """
    sign = torch.sign(w)
    mag = w.abs().clamp(min=1e-8)
    exponent = torch.clamp(torch.floor(torch.log2(mag)), -6, 8)      # 4-bit exponent range
    mantissa_steps = 2 ** 3                                          # 3 mantissa bits
    frac = mag / (2 ** exponent)                                      # in [1, 2)
    frac_q = torch.round((frac - 1) * mantissa_steps) / mantissa_steps + 1
    return sign * frac_q * (2 ** exponent)

# A weight distribution with real outliers: 99% small, 1% ten to twenty times larger --
# this is the shape that makes the precision/range tradeoff visible.
W_wide = torch.randn(20000) * 0.4
n_outliers = 200
W_wide[:n_outliers] *= (10 + 10 * torch.rand(n_outliers))

for name, fn in [("INT8 (uniform grid)", quantize_int8_symmetric), ("FP8 E4M3 (floating grid)", quantize_fp8_e4m3)]:
    w_q = fn(W_wide)
    rel_err = (W_wide - w_q).norm() / W_wide.norm()
    small = W_wide[n_outliers:]
    small_q = w_q[n_outliers:]
    small_rel_err = (small - small_q).norm() / small.norm()
    print(f"{name:26s}  overall error {rel_err:6.2%}   error on the 99% small values {small_rel_err:6.2%}")


INT8 (uniform grid)         overall error  5.48%   error on the 99% small values 10.58%
FP8 E4M3 (floating grid)    overall error  2.40%   error on the 99% small values  2.67%


INT8's single fixed grid has to stretch to cover the outliers, so every "normal" value pays for the tail's dynamic range. FP8's floating-point grid gets denser near zero automatically, so the bulk of the distribution keeps its precision *without needing a calibration step to find and protect outliers* — which is exactly why FP8 quantization is usually "dynamic" (computed on the fly, per tensor or per token, with zero calibration data) while good INT4 needs AWQ/GPTQ's calibration machinery to be competitive. The cost: FP8 only pays off on hardware with native FP8 support (Hopper-generation GPUs — H100/H200 — or newer), and its 8 bits are roughly 2x the memory of INT4 for a smaller quality loss.

## GGUF and Ollama's quant tags

Everything above targets GPU serving frameworks (vLLM, TensorRT-LLM). **GGUF** is llama.cpp's own format — the one Ollama pulls under the hood — and it's built for a different goal: run well on a CPU or a consumer GPU with limited VRAM, not squeeze the last percent of throughput out of a datacenter GPU. GGUF's "K-quants" use the same per-group idea you implemented above (`quantize_int4_per_group`), plus a twist: they store *some* layers or channels at higher precision within the same file, using heuristics about which weights matter most.

| Tag | Bits/weight (effective) | What it means |
|---|---|---|
| `Q4_0` | ~4.5 | Plain per-block INT4, no mixed precision — the oldest, coarsest GGUF format |
| `Q4_K_M` | ~4.8 | K-quant: most weights at 4-bit, the most sensitive layers bumped to 6-bit. The community default for local 4-bit inference |
| `Q5_K_M` | ~5.7 | Same idea, one bit higher across the board — noticeably better quality, noticeably bigger |
| `Q8_0` | 8 | Uniform 8-bit, no mixed precision needed (8 bits already preserves almost everything) — under 0.5% quality loss vs. fp16, at half the file size |

## Decision table: which format for which stack

| You're serving with... | Use this format | Where it comes from |
|---|---|---|
| vLLM / SGLang, INT4, general-purpose | **AWQ** | `llm-compressor`, or a pre-quantized checkpoint tagged `-AWQ` |
| vLLM / SGLang, INT4, squeezing out the last bit of perplexity | **GPTQ** | `llm-compressor`'s GPTQ path, or a `-GPTQ` checkpoint |
| vLLM / SGLang on H100/H200+, want max throughput | **FP8** | `llm-compressor`'s dynamic FP8 recipe — no calibration set needed |
| Ollama / llama.cpp, laptop or consumer GPU | **GGUF** (`Q4_K_M` default, `Q8_0` if you have the RAM) | `ollama pull model:q4_K_M`, or convert with llama.cpp's `convert_hf_to_gguf.py` |

The pattern to remember: **AWQ/GPTQ/FP8 are GPU-server formats; GGUF is the local/edge format.** They solve overlapping math with incompatible file layouts — a vLLM server cannot load a `.gguf` file, and `ollama` cannot load an AWQ checkpoint.


In [5]:
# Guarded live check: if an Ollama daemon is reachable, show what its quant tags cost
# in practice. Fully optional -- everything above already taught the concept without it.
def ollama_up(host="http://localhost:11434", timeout=0.5):
    import urllib.request
    try:
        urllib.request.urlopen(f"{host}/api/tags", timeout=timeout)
        return True
    except Exception:
        return False

HAS_OLLAMA = ollama_up()
print(f"Local Ollama daemon detected: {HAS_OLLAMA}")

if HAS_OLLAMA:
    import ollama
    models = ollama.list()
    names = [m.model for m in models.models]
    print(f"Locally pulled models: {names}")
    print("Pull a quant pair to compare yourself, e.g.:")
    print("  ollama pull qwen2.5:0.5b-instruct-q4_K_M")
    print("  ollama pull qwen2.5:0.5b-instruct-q8_0")
    print("then time a fixed prompt against each with `ollama run <tag> --verbose`.")
else:
    print("[no local Ollama daemon] If one were running, `ollama show <model>` would print")
    print("the exact quantization tag and file size for whatever you've pulled -- the same")
    print("Q4_K_M / Q8_0 tradeoff from the table above, on a real checkpoint.")


Local Ollama daemon detected: False
[no local Ollama daemon] If one were running, `ollama show <model>` would print
the exact quantization tag and file size for whatever you've pulled -- the same
Q4_K_M / Q8_0 tradeoff from the table above, on a real checkpoint.


## Exercises

In [6]:
# Exercise 1 (Warm-up): How much does group size matter for a REALLY outlier-heavy row?
# Task: Make a weight row with 20 (not 8) outlier channels, each 20x (not 12x) larger than
#       the rest. Re-run quantize_int4_per_group at group_size=512, 128, 32, 8 and print the
#       relative error for each. At what group size does the error stop improving much?
# Hint: Reuse quantize_int4_per_group exactly as defined above -- only the input W changes.

# YOUR CODE HERE


In [7]:
# Exercise 2 (Apply): Sweep AWQ's alpha to find the actual optimum
# Task: awq_style_error(W_layer, X_calib, alpha) was evaluated at 7 alphas above. Evaluate
#       it on a finer grid of 15 alphas between 0 and 0.6, plot error vs alpha, and print
#       which alpha minimizes error. This is literally what AWQ's per-layer calibration
#       search does (over a small grid, per layer, using held-out perplexity instead of
#       this toy relative error).
# Hint: np.linspace(0, 0.6, 15); collect (alpha, error) pairs; plt.plot then plt.show().

# YOUR CODE HERE


In [8]:
# Exercise 3 (Extend): Does GPTQ's compensation still help with independent columns?
# Task: Build X_indep with i.i.d. random columns (no repeat_interleave correlation trick),
#       compute H_indep = X_indep.T @ X_indep, and compare naive_sequential_quantize vs
#       gptq_style_quantize output error on the same w_true. Explain in a comment why the
#       gap between them should shrink toward zero.
# Hint: torch.randn(300, N_FEATS) directly, no repeat_interleave -- that removes the
#       cross-column correlation GPTQ's H^-1 term needs to have information content.

# YOUR CODE HERE


<details>
<summary>Show solutions</summary>

```python
# Exercise 1
W_extreme = torch.randn(1, IN_FEATURES) * 0.5
extreme_idx = rng.choice(IN_FEATURES, size=20, replace=False)
W_extreme[0, extreme_idx] *= 20
for group_size in [512, 128, 32, 8]:
    W_deq = quantize_int4_per_group(W_extreme, group_size=group_size)
    rel_err = (W_extreme - W_deq).norm() / W_extreme.norm()
    print(f"group_size={group_size:4d}  relative error: {rel_err:.1%}")
# Error keeps dropping as groups shrink but flattens out once groups are small enough
# that each one holds at most a couple of outliers -- past that point you're just paying
# more scale-storage overhead for no further accuracy gain.

# Exercise 2
alphas = np.linspace(0, 0.6, 15)
errors = [awq_style_error(W_layer, X_calib, alpha=a).item() for a in alphas]
plt.plot(alphas, errors, marker="o")
plt.xlabel("alpha (protection strength)"); plt.ylabel("output relative error")
plt.title("AWQ's per-layer alpha search"); plt.show()
best_alpha = alphas[int(np.argmin(errors))]
print(f"Best alpha: {best_alpha:.2f}  (error {min(errors):.2%})")
# Error decreases then rises again -- exactly why AWQ searches a grid instead of
# picking a single fixed alpha for every layer.

# Exercise 3
X_indep = torch.randn(300, N_FEATS)
H_indep = X_indep.T @ X_indep
w_naive_i = naive_sequential_quantize(w_true)
w_gptq_i = gptq_style_quantize(w_true, H_indep)
y_true_i = X_indep @ w_true
err_naive_i = (y_true_i - X_indep @ w_naive_i).norm() / y_true_i.norm()
err_gptq_i = (y_true_i - X_indep @ w_gptq_i).norm() / y_true_i.norm()
print(f"Naive: {err_naive_i:.2%}   GPTQ-style: {err_gptq_i:.2%}")
# With independent columns H is (nearly) diagonal, so its inverse is too -- the
# off-diagonal H_inv[0,k] terms GPTQ's compensation relies on are close to zero, so
# there's little correlation left to exploit and it collapses toward naive per-column
# rounding. GPTQ's advantage comes entirely from real weight matrices having
# correlated columns.
```
</details>


## Key Takeaways
- Naive round-to-nearest quantization (notebook 10) treats every weight equally; real weight matrices have a few **salient channels** (identified by large activation magnitude) that deserve more precision than the rest.
- **AWQ** protects salient channels by scaling weights up before quantizing and activations down to compensate — a calibration-search over one scalar per layer, fast to produce, no gradients. Too much scaling backfires, which is why AWQ searches a small grid instead of maximizing protection blindly.
- **GPTQ** quantizes column-by-column and compensates not-yet-quantized columns for each column's rounding error, using the *inverse* of the calibration data's correlation structure (`H^-1`, from `H = X^TX`) — its advantage disappears if columns are independent.
- **FP8** spends its 8 bits on dynamic range (floating exponent) instead of uniform precision, so it needs no calibration data and handles outliers gracefully — but requires Hopper-or-newer hardware to pay off.
- **GGUF/Ollama tags** (`Q4_K_M`, `Q8_0`, ...) solve the same problem for CPU/consumer-GPU serving with llama.cpp; they are a different file format from AWQ/GPTQ/FP8 and the two families don't load into each other's servers.

## What's Next
Notebook **13c — KV Cache Eviction** turns from *the model's weights* to *the KV cache* as the thing running out of memory: once you've fit a model onto a GPU, a long enough conversation can still exhaust it, and eviction policies decide what to throw away.
